In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

/home/dongmin/userdata/dongmin/sqomr/modules/omrdp


/home/dongmin/userdata/dongmin/sqomr/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import shutil
from pathlib import Path
from collections import defaultdict
from itertools import groupby

import math
import random

import json
import csv
import strictyaml as syaml

from tqdm import tqdm
import matplotlib.pyplot as plt

import cv2
import numpy as np
import music21
import musicdiff as md
import converter21 as c21

import xml.etree.ElementTree as ET
from modules.lmxe.lmxe import linearize_lmxe, delinearize_lmxe
from modules.lmxe.lmxe.utils import split_tokens_by_key_token, find_nth_token_index
from modules.lmxe.lmxe.Linearizer import LMXELinearizer
from modules.lmxe.lmxe.Delinearizer import LMXEDelinearizer
from modules.lmxe.lmxe.LMXEFile import LMXEFile, LMXEMetadata, get_score_type
from modules.lmxe.lmxe.symbolic.MxlFile import MusicXMLFile
from modules.lmxe.lmxe.symbolic.part_to_score import parts_to_score
from modules.lmxe.lmxe.symbolic.debug_compare import compare_parts

from omrdp.utils import load_ossq_metadata
from omrdp.utils.image import crop_white_space

from IPython.core.magic import register_cell_magic
@register_cell_magic
def skip(line, cell):
  return

/home/dongmin/userdata/dongmin/sqomr/.venv/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Load metadata

In [3]:
dataset_dir = Path.home() / 'userdata' / 'open-score-string-quartets'
dataset_data_dir = dataset_dir / 'data'
dataset_score_dir = dataset_dir / 'scores'

dataset_dir.exists()

True

In [4]:
metadata = load_ossq_metadata(dataset_data_dir / 'scores.yaml')
len(metadata)

122

In [5]:
temp_dir = Path.cwd() / 'temp'
temp_dir.mkdir(exist_ok=True)

## Load excluded segments

In [10]:
excluded_segments = [
  ('systemwise', 'synthetic', 'lmxe'), 
  ('systemwise', 'synthetic', 'krn'), 
  ('systemwise', 'synthetic', 'abc'), 
  ('systemwise', 'scanned', 'lmxe'), 
  ('systemwise', 'scanned', 'krn'), 
  ('systemwise', 'scanned', 'abc'), 
  ('partwise', 'synthetic', 'lmxe'), 
  ('partwise', 'synthetic', 'abc'), 
  ('partwise', 'scanned', 'lmxe'), 
  ('partwise', 'scanned', 'abc'), 
]

excluded_segments = [
  (
    level, img_type, encoding,
    set(
      l.strip()
      for l in (dataset_dir / f'excluded_segments_{img_type}_{encoding}_{level}.txt').read_text().splitlines()
      if l.strip()
    )
  )
  for level, img_type, encoding in excluded_segments
]

for level, img_type, encoding, segments in excluded_segments:
  print(f"{level} {img_type} {encoding}: {len(segments)} segments")

systemwise synthetic lmxe: 3 segments
systemwise synthetic krn: 735 segments
systemwise synthetic abc: 2806 segments
systemwise scanned lmxe: 0 segments
systemwise scanned krn: 613 segments
systemwise scanned abc: 2454 segments
partwise synthetic lmxe: 4 segments
partwise synthetic abc: 2067 segments
partwise scanned lmxe: 1 segments
partwise scanned abc: 1816 segments


In [32]:
total_exclude_segments = {
  ('systemwise', 'synthetic'): set(),
  ('systemwise', 'scanned'): set(),
  ('partwise', 'synthetic'): set(),
  ('partwise', 'scanned'): set(),
}

for level, img_type, encoding, segments in excluded_segments:
  total_exclude_segments[(level, img_type)].update(segments)

for (level, img_type), segments in total_exclude_segments.items():
  if level == 'systemwise':
    segments.update((
      item[:-2] if len(item.split(':')) > 3 else item
      for item in total_exclude_segments[('partwise', img_type)]
    ))
  
  if level == 'partwise':
    segments.update((
      item
      for item in total_exclude_segments[('systemwise', img_type)]
    ))
  
  exclusion_list_len = len(segments)
  
  lmxe_paths = []
  for score_id, body in metadata.items():
    sqid = body['sqid']
    sq_dir = dataset_score_dir / body['path']
    l_p_l = sq_dir.glob(f'lmxe/{img_type}/{level}/{sqid}*.lmxe')
    l_p_l = sorted(l_p_l)
    lmxe_paths += (l_p_l)
  
  num_total_segments = len(lmxe_paths)
  
  if level == 'systemwise':
    lmxe_paths = [
      p
      for p in lmxe_paths
      if p.stem not in segments
    ]
  if level == 'partwise':
    lmxe_paths = [
      p
      for p in lmxe_paths
      if (
        p.stem[:-2] not in segments and
        p.stem not in segments
      )
    ]
  
  num_included = len(lmxe_paths)
  
  num_excluded = num_total_segments - num_included
  
  
  print(f"{level} {img_type}")
  print(f"  Total segments: {num_total_segments}")
  print(f"  Exclusion list length: {exclusion_list_len}")
  print(f"  Excluded segments: {num_excluded}")
  print(f"  Included segments: {num_included}", f"({num_included/4:.2f})" if level == 'partwise' else "")
  print(f"  Exclusion rate: {num_excluded / num_total_segments * 100:.2f}%")
  print()

systemwise synthetic
  Total segments: 13241
  Exclusion list length: 3228
  Excluded segments: 3225
  Included segments: 10016 
  Exclusion rate: 24.36%

systemwise scanned
  Total segments: 11305
  Exclusion list length: 2796
  Excluded segments: 2796
  Included segments: 8509 
  Exclusion rate: 24.73%

partwise synthetic
  Total segments: 52960
  Exclusion list length: 5295
  Excluded segments: 12896
  Included segments: 40064 (10016.00)
  Exclusion rate: 24.35%

partwise scanned
  Total segments: 45212
  Exclusion list length: 4612
  Excluded segments: 11180
  Included segments: 34032 (8508.00)
  Exclusion rate: 24.73%



In [ ]:
excluded_segments_file_names = [
  'lmxe_systemwise', 
  'lmxe_partwise', 
  'krn_systemwise', 
  'abc_systemwise', 
  'abc_partwise', 
]

excluded_segments_synthetic = set()
for fname in excluded_segments_file_names:
  fpath = dataset_dir / f'excluded_segments_synthetic_{fname}.txt'
  lines = fpath.read_text(encoding='utf-8').splitlines()
  for line in lines:
    line = line.strip()
    
    if not line:
      continue
    
    if len(line.split(':')) > 3:
      line = line[:-2]
    
    excluded_segments_synthetic.add(line)

excluded_segments_scanned = set()
for fname in excluded_segments_file_names:
  fpath = dataset_dir / f'excluded_segments_scanned_{fname}.txt'
  lines = fpath.read_text(encoding='utf-8').splitlines()
  for line in lines:
    line = line.strip()
    
    if not line:
      continue
    
    if len(line.split(':')) > 3:
      line = line[:-2]
    
    excluded_segments_scanned.add(line)

print('Number of excluded segments')
print('synthetic:', len(excluded_segments_synthetic))
print('scanned:', len(excluded_segments_scanned))

lmxe_partwise
False
abc_partwise
True
lmxe_partwise
False
abc_partwise
True
Number of excluded segments
synthetic: 3231
scanned: 2796
